# Stim → QIR

Compile [Stim](https://github.com/quantumlib/Stim) circuits to QIR and simulate them with `qdk.stim`.

- `stim.compile(src, None)` → `(qir, noise)`
- `stim.run(src, shots=..., type="clifford")` → per-shot results

In [1]:
from qdk import stim
from qsharp_widgets import Histogram

## Basics

Compile a Bell pair to QIR.

In [2]:
bell = """H 0
CX 0 1
MR 0 1
"""

qir, _ = stim.compile(bell, None)
print(qir)

define i64 @ENTRYPOINT__main() #0 {
  call void @__quantum__rt__initialize(ptr null)
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__cx__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 1 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 1 to ptr), ptr inttoptr (i64 1 to ptr))
  call void @__quantum__rt__array_record_output(i64 2, ptr null)
  call void @__quantum__rt__result_record_output(ptr inttoptr (i64 0 to ptr), ptr null)
  call void @__quantum__rt__result_record_output(ptr inttoptr (i64 1 to ptr), ptr null)
  ret i64 0
}

declare void @__quantum__qis__cx__body(ptr, ptr)
declare void @__quantum__qis__mresetz__body(ptr, ptr)
declare void @__quantum__rt__result_record_output(ptr, ptr)
declare void @__quantum__rt__array_record_output(i64, ptr)
declare void @__quantum__rt__initialize(ptr)
declare void @__quantum__qis

In [3]:
# Entangled: only 00 and 11 appear.
Histogram(["".join(map(str, s)) for s in stim.run(bell, shots=2000, type="clifford")])

## Preselection

`#!preselect_expect` retries until the measurement matches, so the recorded result is always that value — a single bar.

In [6]:
# Force the measurement to 1: every shot reports 1.
preselect = """#!preselect_begin
H 0
MR 0
#!preselect_expect 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(preselect, shots=2000, type="clifford")])

In [ ]:
## Noise channels

### Correlated error

`X0 X1` fire together → only `00` and `11`.

In [9]:
correlated = """CORRELATED_ERROR(0.2) X0 X1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(correlated, shots=5000, type="clifford")])

### Pauli error

Independent `X` on each qubit (p = 0.1).

In [13]:
xerr = """X_ERROR(0.1) 0 1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(xerr, shots=5000, type="clifford")])

### Loss

Lost qubits show up as `L` (p = 0.15).

In [17]:
loss = """LOSS_ERROR(0.15) 0 1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(loss, shots=5000, type="clifford")])

### Loss in a correlated error

Branches mix loss (`L`) and Pauli (`X`) terms.

In [21]:
mixed = """CORRELATED_ERROR(0.1) L0
ELSE_CORRELATED_ERROR(0.1) L1
ELSE_CORRELATED_ERROR(0.1) L0 X1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(mixed, shots=5000, type="clifford")])

### Depolarizing

`DEPOLARIZE1(p)`: one of 3 Paulis per qubit. `DEPOLARIZE2(p)`: one of 15 two-qubit Paulis.

In [22]:
dep1 = """DEPOLARIZE1(0.3) 0 1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(dep1, shots=5000, type="clifford")])

In [23]:
dep2 = """DEPOLARIZE2(0.3) 0 1
MR 0 1
"""
Histogram(["".join(map(str, s)) for s in stim.run(dep2, shots=5000, type="clifford")])